# 3-Stage 평가서버 호환 베이스라인 — 학습

공개 예제 각 5건으로 모델을 학습하고 실제 참가자 제출구조와 동일한 경로에 체크포인트를 저장합니다.

In [ ]:
%pip install -r requirements.txt

## 1. 라이브러리·모델 구조·학습함수

In [ ]:
from pathlib import Path
import os, random, shutil
import cv2, numpy as np, pandas as pd, torch
from PIL import Image
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.video import mvit_v2_s


In [ ]:
ROOT=Path.cwd(); DATA=ROOT/'data'; MODEL=ROOT/'model'
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS=int(os.getenv('EPOCHS','1'))

SIZE=224
S1_MEAN=torch.tensor([0.45,0.45,0.45])[:,None,None,None]
S1_STD=torch.tensor([0.225,0.225,0.225])[:,None,None,None]
S3_MEAN=torch.tensor([0.45,0.45,0.45])[:,None,None]
S3_STD=torch.tensor([0.225,0.225,0.225])[:,None,None]
torch.manual_seed(20260825); random.seed(20260825)

In [ ]:
def _video_frames(path):
    cap=cv2.VideoCapture(str(path)); out=[]
    while True:
        ok,bgr=cap.read()
        if not ok: break
        out.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    cap.release()
    if not out: raise ValueError(f'cannot decode: {path}')
    return out

def _crop_tensor(rgb,size=224):
    h,w=rgb.shape[:2]; scale=size/min(h,w)
    nh,nw=max(size,round(h*scale)),max(size,round(w*scale))
    rgb=cv2.resize(rgb,(nw,nh),interpolation=cv2.INTER_AREA)
    y,x=(nh-size)//2,(nw-size)//2
    return torch.from_numpy(rgb[y:y+size,x:x+size].copy()).permute(2,0,1).float()/255

def _clip(path,n=16,center=None):
    frames=_video_frames(path); total=len(frames)
    if center is None: idx=np.linspace(0,total-1,n).round().astype(int)
    else: idx=np.clip(center-n//2+np.arange(n),0,total-1)
    x=torch.stack([_crop_tensor(frames[int(i)]) for i in idx],1)
    return x,total

In [ ]:
class Stage1MViT(nn.Module):
    def __init__(self):
        super().__init__(); self.net=mvit_v2_s(weights=None)
        self.net.head[1]=nn.Linear(self.net.head[1].in_features,2)
    def forward(self,x): return self.net(x)

class Stage2Temporal(nn.Module):
    def __init__(self):
        super().__init__()
        self.r=nn.GRU(512,192,2,batch_first=True,bidirectional=True,dropout=0.15)
        self.tc=nn.Linear(384,1); self.te=nn.Linear(384,1)
        self.scene=nn.Sequential(nn.Linear(768,192),nn.ReLU(),nn.Dropout(0.2),nn.Linear(192,4))
    def logits(self,x):
        h,_=self.r(x)
        return self.tc(h).squeeze(-1),self.te(h).squeeze(-1),h
    def forward(self,x):
        collision,entry,h=self.logits(x)
        ci,ei=collision.argmax(1),entry.argmax(1); b=torch.arange(len(h),device=h.device)
        return ci,ei,self.scene(torch.cat([h[b,ci],h[b,ei]],1))

class Stage3MViT(nn.Module):
    def __init__(self):
        super().__init__(); self.backbone=mvit_v2_s(weights=None)
        dim=self.backbone.head[1].in_features; self.backbone.head=nn.Identity()
        self.accel=nn.Linear(dim,4); self.steer=nn.Linear(dim,3)
    def forward(self,x):
        z=self.backbone(x); return self.accel(z),self.steer(z)

In [ ]:
def fit_stage1():
    out=MODEL/'stage1'; out.mkdir(parents=True,exist_ok=True)
    df=pd.read_csv(DATA/'stage1/labels.csv')
    model=Stage1MViT().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),1e-4)
    for _ in range(EPOCHS):
        model.train()
        for r in df.sample(frac=1,random_state=20260825).itertuples():
            x,_=_clip(DATA/'stage1'/r.path,16); x=(x-S1_MEAN)/S1_STD
            y=torch.tensor([0 if r.label=='ORIGINAL' else 1],device=DEVICE)
            loss=nn.functional.cross_entropy(model(x[None].to(DEVICE)),y)
            opt.zero_grad(); loss.backward(); opt.step()
    # 실제 inference.py는 래퍼가 아닌 mvit_v2_s 본체에 직접 로드한다.
    torch.save({'model':model.net.state_dict(),'size':224,'frames':16},out/'best.pt')

def _resnet_backbone():
    try: model=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    except Exception:
        print('경고: ImageNet 가중치를 받지 못해 weights=None으로 진행합니다.')
        model=resnet18(weights=None)
    return model

def fit_stage2():
    out=MODEL/'stage2'; out.mkdir(parents=True,exist_ok=True)
    df=pd.read_csv(DATA/'stage2/labels.csv')
    backbone=_resnet_backbone(); torch.save(backbone.state_dict(),out/'resnet18-f37072fd.pth')
    backbone.fc=nn.Identity(); backbone.to(DEVICE).eval()
    transform=ResNet18_Weights.IMAGENET1K_V1.transforms()
    sequences=[]
    with torch.inference_mode():
        for r in df.itertuples():
            frames=_video_frames(DATA/'stage2'/r.path); batches=[]
            for start in range(0,len(frames),64):
                x=torch.stack([transform(Image.fromarray(a)) for a in frames[start:start+64]]).to(DEVICE)
                batches.append(backbone(x).float().cpu())
            sequences.append((torch.cat(batches),min(int(r.t_collision),len(frames)-1)))
    temporal=Stage2Temporal().to(DEVICE); opt=torch.optim.AdamW(temporal.parameters(),2e-4)
    for _ in range(max(1,EPOCHS)):
        temporal.train()
        for seq,target in sequences:
            collision,_,_=temporal.logits(seq[None].to(DEVICE))
            loss=nn.functional.cross_entropy(collision,torch.tensor([target],device=DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
    # 공개 CCD 5건에는 충돌 구간만 공식 주석이 있어 나머지 헤드는 구조 확인용이다.
    torch.save({'model':temporal.state_dict()},out/'best.pt')

def fit_stage3():
    out=MODEL/'stage3'; out.mkdir(parents=True,exist_ok=True)
    df=pd.read_csv(DATA/'stage3/labels.csv')
    amap={'ACCELERATING':0,'DECELERATING':1,'CONSTANT':2,'STOPPED':3}
    smap={'LEFT':0,'STRAIGHT':1,'RIGHT':2}
    model=Stage3MViT().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),1e-4)
    for _ in range(EPOCHS):
        model.train()
        for r in df.itertuples():
            x,_=_clip(DATA/'stage3/videos'/f'{r.ID}.mp4',16,int(r.frame_index))
            x=(x-S3_MEAN[:,None,:,:])/S3_STD[:,None,:,:]
            a,s=model(x[None].to(DEVICE))
            loss=nn.functional.cross_entropy(a,torch.tensor([amap[r.accel_label]],device=DEVICE))
            loss+=nn.functional.cross_entropy(s,torch.tensor([smap[r.steer_label]],device=DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
    torch.save({'model':model.state_dict()},out/'best.pt')


## 2. Stage 1·2·3 학습

In [ ]:
print('device:',DEVICE)
fit_stage1(); print('Stage 1 완료')
fit_stage2(); print('Stage 2 완료')
fit_stage3(); print('Stage 3 완료')

In [ ]:
for p in sorted((ROOT/'model').rglob('*')):
    if p.is_file(): print(p.relative_to(ROOT),f'{p.stat().st_size/1024**2:.1f} MB')